# Generative AI Dependency and Perceived Negative Effects on Problem-Solving

**A cross-sectional study of university students**  
**Author:** Abe Tayo Noah  
**Program:** DataraFlow Cohort 2

## Research overview
This notebook analyzes survey responses from U.S. and Indonesian university students to examine associations between generative AI dependency and **self-reported perceived negative effects** on problem-solving. The analyses are observational and **do not establish causality or objectively measure cognitive decline**.

## Reproducibility
1. Install packages from `requirements.txt` in the repository root.
2. Obtain the source workbook from the original dataset record (see `data/README.md`).
3. Save the workbook as `data/When Culture Meets AI - Survey Dataset.xlsx`.
4. Run this notebook from top to bottom.

**Source:** Pertiwi & Adiningrum (2026), *When Culture Meets AI: A Cross-Cultural Study of Generative AI Dependency Among U.S. and Indonesian Business Students*. https://doi.org/10.5281/zenodo.21013786

---


#### Dataset Loading

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from pathlib import Path

# Works when launched from the repository root or notebooks/ folder.
root = Path.cwd()
if not (root / "data").is_dir() and (root.parent / "data").is_dir():
    root = root.parent
file_path = root / "data" / "When Culture Meets AI - Survey Dataset.xlsx"
if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {file_path}. See data/README.md for download instructions."
    )

survey = pd.read_excel(file_path, sheet_name="Survey Data")
codebook = pd.read_excel(file_path, sheet_name="Codebook")

print("Survey dataset shape:", survey.shape)
print("Codebook shape:", codebook.shape)

display(survey.head())

Survey dataset shape: (239, 41)
Codebook shape: (41, 6)


,RespondentID,Country,Age_Range,Gender,Nationality,Field_of_Study,Level_of_Study,Year_of_Study,GenAI_Experience,GenAI_Tools_Used,...,CT10,CT11,TP1,TP2,TP3,TP4,TP5,TP6,TP7,AnalyticSample
0,USA001,USA,20 - 22,Female,American,Economics Related Majors,Undergraduate (Bachelor's Degree),Junior (3rd-year student),More than 2 years,ChatGPT,...,4.0,4.0,4.0,3.0,4.0,4.0,4.0,3.0,4.0,Yes
1,USA002,USA,20 - 22,Female,American,Business Related Majors,Undergraduate (Bachelor's Degree),Senior (final-year student),1 - 2 years,ChatGPT;Gemini AI,...,3.0,4.0,5.0,5.0,5.0,4.0,4.0,4.0,4.0,Yes
2,USA003,USA,23 - 25,Female,American,Management Related Majors,Undergraduate (Bachelor's Degree),Sophomore (2nd-year student),1 - 2 years,ChatGPT;Gemini AI,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,4.0,Yes
3,USA004,USA,23 - 25,Male,American,Economics Related Majors,Undergraduate (Bachelor's Degree),Junior (3rd-year student),1 - 2 years,ChatGPT;Gemini AI,...,5.0,5.0,5.0,4.0,5.0,5.0,5.0,5.0,5.0,Yes
4,USA005,USA,20 - 22,Prefer not to say,American,Business Related Majors,Undergraduate (Bachelor's Degree),Senior (final-year student),6 months - 1 year,ChatGPT;Gemini AI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No


#### Dataset Inspection

In [5]:
print("Dataset information:")
survey.info()

print("\nColumn names:")
print(survey.columns.tolist())

print("\nMissing values:")
print(survey.isnull().sum().to_string())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239 entries, 0 to 238
Data columns (total 41 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   RespondentID      239 non-null    object 
 1   Country           239 non-null    object 
 2   Age_Range         239 non-null    object 
 3   Gender            239 non-null    object 
 4   Nationality       239 non-null    object 
 5   Field_of_Study    239 non-null    object 
 6   Level_of_Study    239 non-null    object 
 7   Year_of_Study     239 non-null    object 
 8   GenAI_Experience  239 non-null    object 
 9   GenAI_Tools_Used  239 non-null    object 
 10  GenAI_Frequency   239 non-null    object 
 11  GAID1             227 non-null    float64
 12  GAID2             227 non-null    float64
 13  GAID3             227 non-null    float64
 14  GAID4             227 non-null    float64
 15  GAID5             227 non-null    float64
 16  GAID6             227 n

In [ ]:
# Checking analytic sample categories
print("Analytic sample values:")
print(survey["AnalyticSample"].value_counts(dropna=False))

# Checking the distribution by country
print("\nRespondents by country:")
print(survey["Country"].value_counts())

# Checking GAID7 missing values
print("\nGAID7 missing values:")
print(survey["GAID7"].isnull().sum())

# Checking  whether missing GAID7 responses align with sample flag
print("\nAnalytic sample vs GAID7 availability:")
print(pd.crosstab(
    survey["AnalyticSample"],
    survey["GAID7"].notna()
))

Analytic sample values:
AnalyticSample
Yes    227
No      12
Name: count, dtype: int64

Respondents by country:
Country
USA          122
Indonesia    117
Name: count, dtype: int64

GAID7 missing values:
12

Analytic sample vs GAID7 availability:
GAID7           False  True 
AnalyticSample              
No                 12      0
Yes                 0    227


#### Preparation and Inspection of research sample

In [7]:

# Selecting respondents eligible for analysis
research = survey.loc[
    survey["AnalyticSample"] == "Yes"
].copy()

print("Research dataset shape:", research.shape)

print("\nEligible respondents by country:")
print(research["Country"].value_counts())

print("\nMissing values among eligible respondents:")
missing = research.isna().sum()
print(missing[missing > 0])

print("\nDuplicate respondent IDs:")
print(research["RespondentID"].duplicated().sum())


Research dataset shape: (227, 41)

Eligible respondents by country:
Country
USA          116
Indonesia    111
Name: count, dtype: int64

Missing values among eligible respondents:
Series([], dtype: int64)

Duplicate respondent IDs:
0


#### Inspect questionnaire variables and codebook

In [8]:

# Identify the survey questions
scale_columns = [
    col for col in research.columns
    if col.startswith(("GAID", "CT", "TP"))
]

print("QUESTIONNAIRE VARIABLES:")
print(scale_columns)

print("\nCODEBOOK COLUMNS:")
print(codebook.columns.tolist())

print("\nCODEBOOK CONTENT:")
display(codebook)

print("\nRESPONSE VALUES FOR EACH QUESTION:")
for col in scale_columns:
    print(f"\n{col}:")
    print(research[col].value_counts().sort_index())


QUESTIONNAIRE VARIABLES:
['GAID1', 'GAID2', 'GAID3', 'GAID4', 'GAID5', 'GAID6', 'GAID7', 'GAID8', 'GAID9', 'GAID10', 'GAID11', 'CT1', 'CT2', 'CT3', 'CT4', 'CT5', 'CT6', 'CT7', 'CT8', 'CT9', 'CT10', 'CT11', 'TP1', 'TP2', 'TP3', 'TP4', 'TP5', 'TP6', 'TP7']

CODEBOOK COLUMNS:
['Variable', 'Description / Item wording', 'Construct', 'Subdimension', 'Response coding / Values', 'Scale source']

CODEBOOK CONTENT:


,Variable,Description / Item wording,Construct,Subdimension,Response coding / Values,Scale source
0,RespondentID,"Anonymous respondent identifier (USA001…, IDN0...",Demographic,—,—,Generated for anonymization
1,Country,National sample,Demographic,—,USA; Indonesia,—
2,Age_Range,Self-reported age band,Demographic,—,17-19; 20-22; 23-25; Above 25,ID 'Diatas 25'→'Above 25'
3,Gender,Self-reported gender,Demographic,—,Male; Female; Prefer not to say,ID Laki-laki→Male; Perempuan→Female
4,Nationality,Self-reported nationality,Demographic,—,American; Indonesia,—
5,Field_of_Study,Business discipline,Demographic,—,Business/Management/Economics/Finance-Accounti...,ID 'Jurusan terkait X'→'X Related Majors'
6,Level_of_Study,Degree level,Demographic,—,Undergraduate (Bachelor's Degree),ID 'Sarjana (S1)'→Undergraduate
7,Year_of_Study,Current year,Demographic,—,Freshman; Sophomore; Junior; Senior,ID 'Tahun 1-4'→Freshman-Senior
8,GenAI_Experience,Length of GenAI use,Demographic,—,Less than 6 months → More than 2 years,ID translated to EN
9,GenAI_Tools_Used,"Tools used (multi-select, ; separated)",Demographic,—,ChatGPT; Gemini; Claude; Perplexity; etc.,"Verbatim, names unchanged"



RESPONSE VALUES FOR EACH QUESTION:

GAID1:
GAID1
1.0    11
2.0    28
3.0    63
4.0    90
5.0    35
Name: count, dtype: int64

GAID2:
GAID2
1.0    19
2.0    35
3.0    58
4.0    77
5.0    38
Name: count, dtype: int64

GAID3:
GAID3
1.0    14
2.0    34
3.0    50
4.0    84
5.0    45
Name: count, dtype: int64

GAID4:
GAID4
1.0    34
2.0    58
3.0    75
4.0    42
5.0    18
Name: count, dtype: int64

GAID5:
GAID5
1.0    28
2.0    49
3.0    70
4.0    62
5.0    18
Name: count, dtype: int64

GAID6:
GAID6
1.0    40
2.0    51
3.0    64
4.0    56
5.0    16
Name: count, dtype: int64

GAID7:
GAID7
1.0    43
2.0    61
3.0    54
4.0    49
5.0    20
Name: count, dtype: int64

GAID8:
GAID8
1.0    52
2.0    69
3.0    48
4.0    34
5.0    24
Name: count, dtype: int64

GAID9:
GAID9
1.0    41
2.0    59
3.0    68
4.0    43
5.0    16
Name: count, dtype: int64

GAID10:
GAID10
1.0    24
2.0    58
3.0    75
4.0    47
5.0    23
Name: count, dtype: int64

GAID11:
GAID11
1.0    60
2.0    65
3.0    58
4.0    29
5.0   

#### Review AI dependency questionnaire wording

In [15]:

from IPython.display import display, HTML

# Select GAID questionnaire items
gaid_codebook = codebook[
    codebook["Variable"].astype(str).str.fullmatch(
        r"GAID(?:[1-9]|10|11)"
    )
].copy()

# Select and rename columns
gaid_table = gaid_codebook[
    [
        "Variable",
        "Description / Item wording",
        "Subdimension"
    ]
].copy()

gaid_table.columns = [
    "Question",
    "Question Wording",
    "Dimension"
]

# Display a readable, wrapped table
display(
    gaid_table.style
    .set_properties(
        subset=["Question Wording"],
        **{
            "white-space": "normal",
            "text-align": "left",
            "min-width": "400px",
            "max-width": "650px"
        }
    )
    .set_properties(
        **{"text-align": "left"}
    )
    .hide(axis="index")
)


Question,Question Wording,Dimension
GAID1,1. My decisions are often influenced by generative AI.,Cognitive Preoccupation
GAID2,"2. I feel an urge to use generative AI, even when it may not be necessary.",Cognitive Preoccupation
GAID3,"3. I look forward to using generative AI, even if it is unrelated to the task at hand.",Cognitive Preoccupation
GAID4,4. My use of generative AI has caused concerns for me.,Negative Consequences
GAID5,5. I have trouble completing work or other responsibilities without generative AI.,Negative Consequences
GAID6,6. I feel less confident in my abilities without generative AI.,Negative Consequences
GAID7,7. My use of generative AI has negatively affected my problem-solving skills or efficiency.,Negative Consequences
GAID8,8. I experience restlessness if I am unable to use generative AI.,Withdrawal
GAID9,9. I feel unsettled or distracted when I cannot use generative AI.,Withdrawal
GAID10,10. I feel a sense of disconnection when I cannot access generative AI.,Withdrawal


#### Constructing research variables

In [ ]:

# Defining the questionnaire dimensions
cognitive_items = ["GAID1", "GAID2", "GAID3"]
withdrawal_items = ["GAID8", "GAID9", "GAID10", "GAID11"]

dependency_items = cognitive_items + withdrawal_items

# Calculate scores
research["Cognitive_Preoccupation"] = research[cognitive_items].mean(axis=1)
research["Withdrawal"] = research[withdrawal_items].mean(axis=1)
research["AI_Dependency_Score"] = research[dependency_items].mean(axis=1)

# Outcome: perceived negative effect on problem-solving
research["Problem_Solving_Impact"] = research["GAID7"]

# Creating a readable descriptive statistics table
variables = [
    "Cognitive_Preoccupation",
    "Withdrawal",
    "AI_Dependency_Score",
    "Problem_Solving_Impact"
]

summary = research[variables].describe().T

summary = summary.rename(columns={
    "count": "Respondents",
    "mean": "Mean",
    "std": "Standard Deviation",
    "min": "Minimum",
    "25%": "25th Percentile",
    "50%": "Median",
    "75%": "75th Percentile",
    "max": "Maximum"
})

display(summary.round(2))

print("\nGAID7 RESPONSE DISTRIBUTION:")
print(
    research["Problem_Solving_Impact"]
    .value_counts()
    .sort_index()
)


,Respondents,Mean,Standard Deviation,Minimum,25th Percentile,Median,75th Percentile,Maximum
Cognitive_Preoccupation,227.0,3.44,0.87,1.0,3.00,3.33,4.00,5.00
Withdrawal,227.0,2.67,0.96,1.0,2.00,2.75,3.25,5.00
AI_Dependency_Score,227.0,3.00,0.79,1.0,2.43,3.00,3.57,4.86
Problem_Solving_Impact,227.0,2.74,1.24,1.0,2.00,3.00,4.00,5.00



GAID7 RESPONSE DISTRIBUTION:
Problem_Solving_Impact
1.0    43
2.0    61
3.0    54
4.0    49
5.0    20
Name: count, dtype: int64


#### Reliability and item diagnostics

In [ ]:

def cronbach_alpha(data):
    items = data.astype(float)
    k = items.shape[1]

    if k < 2:
        return np.nan

    item_variances = items.var(axis=0, ddof=1)
    total_variance = items.sum(axis=1).var(ddof=1)

    if total_variance == 0:
        return np.nan

    return (k / (k - 1)) * (
        1 - item_variances.sum() / total_variance
    )

# Define item groups
cognitive_items = ["GAID1", "GAID2", "GAID3"]
withdrawal_items = [
    "GAID8", "GAID9", "GAID10", "GAID11"
]
dependency_items = cognitive_items + withdrawal_items

# Calculate reliability
reliability_results = pd.DataFrame({
    "Scale": [
        "Cognitive Preoccupation",
        "Withdrawal",
        "Proposed Seven-Item Predictor"
    ],
    "Number of Items": [3, 4, 7],
    "Cronbach Alpha": [
        cronbach_alpha(research[cognitive_items]),
        cronbach_alpha(research[withdrawal_items]),
        cronbach_alpha(research[dependency_items])
    ]
})

display(reliability_results.round(3))

# Corrected item-total correlations
diagnostics = []

for item in dependency_items:
    other_items = [
        col for col in dependency_items
        if col != item
    ]

    corrected_total = research[other_items].sum(axis=1)

    diagnostics.append({
        "Item": item,
        "Mean": research[item].mean(),
        "Standard Deviation": research[item].std(),
        "Corrected Item-Total Correlation":
            research[item].corr(corrected_total)
    })

item_diagnostics = pd.DataFrame(diagnostics)

display(item_diagnostics.round(3))

# Correlation between the two subdimensions
print(
    "Correlation between cognitive preoccupation and withdrawal:",
    round(
        research["Cognitive_Preoccupation"].corr(
            research["Withdrawal"]
        ),
        3
    )
)


,Scale,Number of Items,Cronbach Alpha
0,Cognitive Preoccupation,3,0.663
1,Withdrawal,4,0.810
2,Proposed Seven-Item Predictor,7,0.801


,Item,Mean,Standard Deviation,Corrected Item-Total Correlation
0,GAID1,3.485,1.049,0.488
1,GAID2,3.352,1.175,0.470
2,GAID3,3.493,1.150,0.404
3,GAID8,2.599,1.280,0.618
4,GAID9,2.709,1.173,0.653
5,GAID10,2.943,1.137,0.562
6,GAID11,2.445,1.197,0.536


Correlation between cognitive preoccupation and withdrawal: 0.451


#### Review Critical Thinking and Task Performance items

In [16]:

# Select CT and TP questionnaire items
ct_tp_codebook = codebook[
    codebook["Variable"].astype(str).str.fullmatch(
        r"(?:CT(?:[1-9]|10|11)|TP[1-7])"
    )
].copy()

# Create a readable table
scoring_table = ct_tp_codebook[
    [
        "Variable",
        "Description / Item wording",
        "Construct",
        "Subdimension",
        "Response coding / Values",
        "Scale source"
    ]
].copy()

scoring_table.columns = [
    "Question",
    "Question Wording",
    "Construct",
    "Dimension",
    "Response Coding",
    "Original Scale"
]

display(
    scoring_table.style
    .set_properties(
        **{
            "white-space": "normal",
            "text-align": "left"
        }
    )
    .hide(axis="index")
)


Question,Question Wording,Construct,Dimension,Response Coding,Original Scale
CT1,1. I usually try to think about the bigger picture during a discussion.,Critical Thinking,Critical Openness,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"
CT2,2. I often use new ideas to shape (modify) the way I do things.,Critical Thinking,Critical Openness,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"
CT3,3. I use more than one source to find out information for myself.,Critical Thinking,Critical Openness,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"
CT4,4. I am often on the lookout for new ideas.,Critical Thinking,Critical Openness,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"
CT5,5. I sometimes find a good argument that challenges some of my firmly held beliefs.,Critical Thinking,Critical Openness,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"
CT6,6. It's important to understand other people's viewpoint on an issue.,Critical Thinking,Critical Openness,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"
CT7,7. It's important to justify the choices I make.,Critical Thinking,Critical Openness,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"
CT8,8. I often re-evaluate my experiences so that I can learn from them.,Critical Thinking,Reflective Skepticism,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"
CT9,9. I usually check the credibility of the source of information before making judgements.,Critical Thinking,Reflective Skepticism,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"
CT10,10. I usually think about the wider implications of a decision before taking action.,Critical Thinking,Reflective Skepticism,1 = Strongly disagree … 5 = Strongly agree,"Critical Thinking Disposition Scale (Sosu, 2013)"


#### Construct CT and TP scores and assess reliability



In [17]:

# Define questionnaire items
openness_items = [f"CT{i}" for i in range(1, 8)]
skepticism_items = [f"CT{i}" for i in range(8, 12)]
ct_items = openness_items + skepticism_items

tp_items = [f"TP{i}" for i in range(1, 8)]

# Confirm values are within the expected 1–5 range
all_items = ct_items + tp_items

invalid = (
    ~research[all_items].isin([1, 2, 3, 4, 5])
).sum()

print("Invalid or missing responses:")
print(invalid[invalid > 0])

# Construct provisional scores
research["Critical_Openness"] = research[
    openness_items
].mean(axis=1)

research["Reflective_Skepticism"] = research[
    skepticism_items
].mean(axis=1)

research["Critical_Thinking_Score"] = research[
    ct_items
].mean(axis=1)

research["Task_Performance_Score"] = research[
    tp_items
].mean(axis=1)

# Reliability analysis
scale_groups = {
    "Critical Openness": openness_items,
    "Reflective Skepticism": skepticism_items,
    "Overall Critical Thinking": ct_items,
    "Task Performance": tp_items
}

results = []

for name, items in scale_groups.items():
    results.append({
        "Scale": name,
        "Items": len(items),
        "Cronbach Alpha": cronbach_alpha(
            research[items]
        ),
        "Mean Score": research[items].mean(axis=1).mean(),
        "Standard Deviation": research[items].mean(axis=1).std()
    })

scale_results = pd.DataFrame(results)

display(scale_results.round(3))

print("\nCorrelation between CT dimensions:")
print(
    round(
        research["Critical_Openness"].corr(
            research["Reflective_Skepticism"]
        ),
        3
    )
)


Invalid or missing responses:
Series([], dtype: int64)


,Scale,Items,Cronbach Alpha,Mean Score,Standard Deviation
0,Critical Openness,7,0.665,4.113,0.501
1,Reflective Skepticism,4,0.637,4.235,0.537
2,Overall Critical Thinking,11,0.788,4.157,0.473
3,Task Performance,7,0.704,4.042,0.554



Correlation between CT dimensions:
0.677


#### Spearman correlation analysis


In [19]:
from scipy.stats import spearmanr

variables = {
    "AI Dependency": "AI_Dependency_Score",
    "Cognitive Preoccupation": "Cognitive_Preoccupation",
    "Withdrawal": "Withdrawal",
    "Perceived Problem-Solving Impact": "Problem_Solving_Impact",
    "Critical Thinking": "Critical_Thinking_Score",
    "Task Performance": "Task_Performance_Score"
}

# Correlation matrix
correlation_data = research[list(variables.values())].copy()
correlation_data.columns = list(variables.keys())

spearman_matrix = correlation_data.corr(method="spearman")

print("SPEARMAN CORRELATION MATRIX")
display(spearman_matrix.round(3))

# Individual correlations with the primary outcome
results = []

outcome = research["Problem_Solving_Impact"]

for label, column in variables.items():
    if column == "Problem_Solving_Impact":
        continue

    rho, p_value = spearmanr(
        research[column],
        outcome
    )

    results.append({
        "Variable": label,
        "Spearman rho": rho,
        "p-value": p_value
    })

correlation_results = pd.DataFrame(results)

print("\nCORRELATIONS WITH PERCEIVED PROBLEM-SOLVING IMPACT")
display(correlation_results.round(4))

SPEARMAN CORRELATION MATRIX


,AI Dependency,Cognitive Preoccupation,Withdrawal,Perceived Problem-Solving Impact,Critical Thinking,Task Performance
AI Dependency,1.000,0.778,0.907,0.387,0.072,0.163
Cognitive Preoccupation,0.778,1.000,0.453,0.158,0.260,0.289
Withdrawal,0.907,0.453,1.000,0.436,-0.061,0.038
Perceived Problem-Solving Impact,0.387,0.158,0.436,1.000,-0.144,-0.081
Critical Thinking,0.072,0.260,-0.061,-0.144,1.000,0.574
Task Performance,0.163,0.289,0.038,-0.081,0.574,1.000



CORRELATIONS WITH PERCEIVED PROBLEM-SOLVING IMPACT


,Variable,Spearman rho,p-value
0,AI Dependency,0.3866,0.0000
1,Cognitive Preoccupation,0.1579,0.0172
2,Withdrawal,0.4364,0.0000
3,Critical Thinking,-0.1436,0.0306
4,Task Performance,-0.0807,0.2260



#### Inspect potential regression control variables

In [20]:
control_variables = [
    "Country",
    "Age_Range",
    "Gender",
    "Year_of_Study",
    "GenAI_Experience",
    "GenAI_Frequency"
]

for column in control_variables:
    print(f"\n{'=' * 55}")
    print(f"{column.upper()}")
    print(f"{'=' * 55}")

    counts = research[column].value_counts(
        dropna=False
    )

    percentages = research[column].value_counts(
        normalize=True,
        dropna=False
    ) * 100

    table = pd.DataFrame({
        "Respondents": counts,
        "Percentage (%)": percentages.round(1)
    })

    display(table)

# Outcome distribution by country
print("\nGAID7 RESPONSES BY COUNTRY")

country_outcome = pd.crosstab(
    research["Country"],
    research["Problem_Solving_Impact"]
)

display(country_outcome)



COUNTRY


,Respondents,Percentage (%)
Country,,
USA,116,51.1
Indonesia,111,48.9



AGE_RANGE


,Respondents,Percentage (%)
Age_Range,,
20 - 22,96,42.3
23 - 25,68,30.0
17 - 19,41,18.1
Above 25,22,9.7



GENDER


...

,Respondents,Percentage (%)
GenAI_Experience,,
1 - 2 years,76,33.5
6 months - 1 year,69,30.4
More than 2 years,62,27.3
Less than 6 months,20,8.8



GENAI_FREQUENCY


,Respondents,Percentage (%)
GenAI_Frequency,,
Often,108,47.6
Sometimes,82,36.1
Always,37,16.3



GAID7 RESPONSES BY COUNTRY


Problem_Solving_Impact,1.0,2.0,3.0,4.0,5.0
Country,,,,,
Indonesia,17,32,29,19,14
USA,26,29,25,30,6


#### Primary ordinal logistic regression



In [22]:

from statsmodels.miscmodels.ordinal_model import OrderedModel

# Create a separate modeling dataset
model_data = research.copy()

# Country indicator
model_data["Indonesia"] = (
    model_data["Country"] == "Indonesia"
).astype(int)

# Ordered year-of-study categories
year_mapping = {
    "Freshman (1st-year student)": 1,
    "Sophomore (2nd-year student)": 2,
    "Junior (3rd-year student)": 3,
    "Senior (final-year student)": 4
}

model_data["Study_Year"] = (
    model_data["Year_of_Study"].map(year_mapping)
)

# Ordered AI-use frequency
frequency_mapping = {
    "Sometimes": 1,
    "Often": 2,
    "Always": 3
}

model_data["AI_Frequency"] = (
    model_data["GenAI_Frequency"].map(
        frequency_mapping
    )
)

# Define the outcome and predictors
outcome = model_data[
    "Problem_Solving_Impact"
].astype(int)

predictor_columns = [
    "AI_Dependency_Score",
    "Indonesia",
    "Study_Year",
    "AI_Frequency"
]

predictors = model_data[
    predictor_columns
].astype(float)

# Verify complete modeling data
print("Missing predictor values:")
print(predictors.isna().sum())

print("\nModel sample size:", len(model_data))

# Fit proportional-odds ordinal logistic regression
model = OrderedModel(
    outcome,
    predictors,
    distr="logit"
)

result = model.fit(
    method="bfgs",
    disp=False,
    maxiter=1000
)

print(result.summary())

# Extract odds ratios and 95% confidence intervals
coefficients = result.params.loc[predictor_columns]
confidence_intervals = result.conf_int().loc[
    predictor_columns
]

odds_ratios = pd.DataFrame({
    "Odds Ratio": np.exp(coefficients),
    "CI Lower": np.exp(confidence_intervals[0]),
    "CI Upper": np.exp(confidence_intervals[1]),
    "p-value": result.pvalues.loc[predictor_columns]
})

print("\nODDS RATIOS AND 95% CONFIDENCE INTERVALS")
display(odds_ratios.round(4))

print("\nModel converged:")
print(result.mle_retvals.get("converged", "Check optimizer output"))


Missing predictor values:
AI_Dependency_Score    0
Indonesia              0
Study_Year             0
AI_Frequency           0
dtype: int64

Model sample size: 227
                               OrderedModel Results                               
Dep. Variable:     Problem_Solving_Impact   Log-Likelihood:                -333.01
Model:                       OrderedModel   AIC:                             682.0
Method:                Maximum Likelihood   BIC:                             709.4
Date:                    Sat, 24 Oct 2026                                         
Time:                            07:36:41                                         
No. Observations:                     227                                         
Df Residuals:                         219                                         
Df Model:                               4                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
--

,Odds Ratio,CI Lower,CI Upper,p-value
AI_Dependency_Score,2.9462,2.0647,4.2040,0.0000
Indonesia,0.8855,0.5476,1.4318,0.6199
Study_Year,0.8919,0.7111,1.1187,0.3224
AI_Frequency,0.7348,0.5182,1.0418,0.0836



Model converged:
True


#### Explore the proportional-odds assumption

In [23]:
import statsmodels.api as sm

# Use the same predictors from Step 15
X = sm.add_constant(predictors)

# Compare binary logistic models at each outcome threshold
threshold_results = []

for threshold in [1, 2, 3, 4]:

    # Outcome: GAID7 greater than threshold
    binary_outcome = (outcome > threshold).astype(int)

    binary_model = sm.Logit(binary_outcome, X)

    binary_result = binary_model.fit(disp=False)

    for variable in predictor_columns:
        threshold_results.append({
            "Threshold": f"GAID7 > {threshold}",
            "Predictor": variable,
            "Coefficient": binary_result.params[variable],
            "Odds Ratio": np.exp(
                binary_result.params[variable]
            ),
            "p-value": binary_result.pvalues[variable]
        })

threshold_table = pd.DataFrame(threshold_results)

print("THRESHOLD-SPECIFIC ODDS RATIOS")
display(
    threshold_table.pivot(
        index="Predictor",
        columns="Threshold",
        values="Odds Ratio"
    ).round(3)
)

print("\nTHRESHOLD-SPECIFIC COEFFICIENTS")
display(
    threshold_table.pivot(
        index="Predictor",
        columns="Threshold",
        values="Coefficient"
    ).round(3)
)


THRESHOLD-SPECIFIC ODDS RATIOS


Threshold,GAID7 > 1,GAID7 > 2,GAID7 > 3,GAID7 > 4
Predictor,,,,
AI_Dependency_Score,3.229,2.828,2.326,3.293
AI_Frequency,0.768,0.743,0.862,0.371
Indonesia,1.129,0.827,0.698,2.124
Study_Year,0.900,0.828,0.828,1.067



THRESHOLD-SPECIFIC COEFFICIENTS


Threshold,GAID7 > 1,GAID7 > 2,GAID7 > 3,GAID7 > 4
Predictor,,,,
AI_Dependency_Score,1.172,1.040,0.844,1.192
AI_Frequency,-0.264,-0.297,-0.148,-0.992
Indonesia,0.121,-0.190,-0.359,0.753
Study_Year,-0.105,-0.189,-0.188,0.065


#### Compare ordinal and multinomial models

In [24]:


import statsmodels.api as sm

# Reuse outcome, predictors and result from Step 15
X_multinomial = sm.add_constant(predictors)

# Fit multinomial logistic regression
multinomial_model = sm.MNLogit(
    outcome,
    X_multinomial
)

multinomial_result = multinomial_model.fit(
    method="newton",
    maxiter=1000,
    disp=False
)

# Compare model fit
comparison = pd.DataFrame({
    "Model": [
        "Ordinal Logistic Regression",
        "Multinomial Logistic Regression"
    ],
    "Log-Likelihood": [
        result.llf,
        multinomial_result.llf
    ],
    "AIC": [
        result.aic,
        multinomial_result.aic
    ],
    "BIC": [
        result.bic,
        multinomial_result.bic
    ]
})

print("MODEL COMPARISON")
display(comparison.round(3))

print("\nMULTINOMIAL MODEL CONVERGED:")
print(
    multinomial_result.mle_retvals.get(
        "converged",
        "Check optimizer output"
    )
)


MODEL COMPARISON


,Model,Log-Likelihood,AIC,BIC
0,Ordinal Logistic Regression,-333.007,682.014,709.413
1,Multinomial Logistic Regression,-326.218,692.437,760.936



MULTINOMIAL MODEL CONVERGED:
True



#### Likelihood-ratio test
##### Ordinal logistic vs multinomial logistic



In [25]:
from scipy.stats import chi2

# Difference in log-likelihoods
lr_statistic = 2 * (
    multinomial_result.llf - result.llf
)

# Difference in number of model parameters
df_difference = (
    len(multinomial_result.params.to_numpy().ravel())
    - len(result.params)
)

# Chi-square p-value
p_value = chi2.sf(
    lr_statistic,
    df_difference
)

lr_results = pd.DataFrame({
    "Statistic": [
        "Likelihood-ratio chi-square",
        "Degrees of freedom",
        "p-value"
    ],
    "Value": [
        lr_statistic,
        df_difference,
        p_value
    ]
})

print("PROPORTIONAL-ODDS MODEL COMPARISON")
display(lr_results.round(5))

print("\nINTERPRETATION:")

if p_value < 0.05:
    print(
        "Evidence that the more flexible multinomial "
        "model fits significantly better."
    )
else:
    print(
        "Insufficient evidence that the multinomial "
        "model fits significantly better."
    )


PROPORTIONAL-ODDS MODEL COMPARISON


,Statistic,Value
0,Likelihood-ratio chi-square,13.57727
1,Degrees of freedom,12.00000
2,p-value,0.32851



INTERPRETATION:
Insufficient evidence that the multinomial model fits significantly better.


#### Sensitivity analysis of AI dependency dimensions


In [26]:

from statsmodels.miscmodels.ordinal_model import OrderedModel

control_columns = [
    "Indonesia",
    "Study_Year",
    "AI_Frequency"
]

sensitivity_models = {
    "Cognitive Preoccupation Only": [
        "Cognitive_Preoccupation"
    ],
    "Withdrawal Only": [
        "Withdrawal"
    ],
    "Both Dimensions": [
        "Cognitive_Preoccupation",
        "Withdrawal"
    ]
}

sensitivity_results = []
model_fit_results = []

for model_name, main_predictors in sensitivity_models.items():

    columns = main_predictors + control_columns

    X_sensitivity = model_data[columns].astype(float)

    sensitivity_model = OrderedModel(
        outcome,
        X_sensitivity,
        distr="logit"
    )

    sensitivity_fit = sensitivity_model.fit(
        method="bfgs",
        disp=False,
        maxiter=1000
    )

    confidence_intervals = sensitivity_fit.conf_int()

    for variable in main_predictors:
        sensitivity_results.append({
            "Model": model_name,
            "Predictor": variable,
            "Odds Ratio": np.exp(
                sensitivity_fit.params[variable]
            ),
            "CI Lower": np.exp(
                confidence_intervals.loc[variable, 0]
            ),
            "CI Upper": np.exp(
                confidence_intervals.loc[variable, 1]
            ),
            "p-value": sensitivity_fit.pvalues[variable]
        })

    model_fit_results.append({
        "Model": model_name,
        "AIC": sensitivity_fit.aic,
        "BIC": sensitivity_fit.bic,
        "Converged": sensitivity_fit.mle_retvals.get(
            "converged",
            False
        )
    })

sensitivity_table = pd.DataFrame(sensitivity_results)
fit_table = pd.DataFrame(model_fit_results)

print("SENSITIVITY ANALYSIS RESULTS")
display(sensitivity_table.round(4))

print("\nMODEL FIT COMPARISON")
display(fit_table.round(3))


SENSITIVITY ANALYSIS RESULTS


,Model,Predictor,Odds Ratio,CI Lower,CI Upper,p-value
0,Cognitive Preoccupation Only,Cognitive_Preoccupation,1.4346,1.0588,1.9437,0.0199
1,Withdrawal Only,Withdrawal,2.7779,2.0765,3.7161,0.0000
2,Both Dimensions,Cognitive_Preoccupation,0.8969,0.6391,1.2587,0.5292
3,Both Dimensions,Withdrawal,2.8905,2.1053,3.9685,0.0000



MODEL FIT COMPARISON


,Model,AIC,BIC,Converged
0,Cognitive Preoccupation Only,714.509,741.909,True
1,Withdrawal Only,668.679,696.078,True
2,Both Dimensions,670.282,701.107,True


#### Robustness to control-variable coding

In [27]:

from statsmodels.miscmodels.ordinal_model import OrderedModel

# Model 1: Unadjusted
X_unadjusted = model_data[
    ["AI_Dependency_Score"]
].astype(float)

# Model 2: Original adjusted specification
X_original = model_data[
    [
        "AI_Dependency_Score",
        "Indonesia",
        "Study_Year",
        "AI_Frequency"
    ]
].astype(float)

# Model 3: Treat year and frequency as categorical
year_dummies = pd.get_dummies(
    model_data["Year_of_Study"],
    prefix="Year",
    drop_first=True,
    dtype=float
)

frequency_dummies = pd.get_dummies(
    model_data["GenAI_Frequency"],
    prefix="Frequency",
    drop_first=True,
    dtype=float
)

X_categorical = pd.concat(
    [
        model_data[
            ["AI_Dependency_Score", "Indonesia"]
        ].astype(float),
        year_dummies,
        frequency_dummies
    ],
    axis=1
)

specifications = {
    "Unadjusted": X_unadjusted,
    "Original Adjusted": X_original,
    "Categorical Adjusted": X_categorical
}

robustness_results = []

for name, X in specifications.items():

    model = OrderedModel(
        outcome,
        X,
        distr="logit"
    )

    fit = model.fit(
        method="bfgs",
        disp=False,
        maxiter=1000
    )

    ci = fit.conf_int().loc["AI_Dependency_Score"]

    robustness_results.append({
        "Model": name,
        "Odds Ratio": np.exp(
            fit.params["AI_Dependency_Score"]
        ),
        "CI Lower": np.exp(ci[0]),
        "CI Upper": np.exp(ci[1]),
        "p-value": fit.pvalues["AI_Dependency_Score"],
        "AIC": fit.aic,
        "BIC": fit.bic,
        "Converged": fit.mle_retvals.get(
            "converged",
            False
        )
    })

robustness_table = pd.DataFrame(robustness_results)

print("ROBUSTNESS TO CONTROL-VARIABLE CODING")
display(robustness_table.round(4))


ROBUSTNESS TO CONTROL-VARIABLE CODING


,Model,Odds Ratio,CI Lower,CI Upper,p-value,AIC,BIC,Converged
0,Unadjusted,2.6527,1.9042,3.6954,0.0,680.3304,697.4552,True
1,Original Adjusted,2.9462,2.0647,4.2040,0.0,682.0138,709.4134,True
2,Categorical Adjusted,2.7972,1.9553,4.0017,0.0,683.5155,721.1899,True


#### Exploratory country interaction analysis

In [28]:

# Use the original adjusted predictors
X_country = model_data[
    [
        "AI_Dependency_Score",
        "Indonesia",
        "Study_Year",
        "AI_Frequency"
    ]
].astype(float).copy()

# Add AI Dependency x Country interaction
X_country["Dependency_x_Indonesia"] = (
    X_country["AI_Dependency_Score"]
    * X_country["Indonesia"]
)

# Fit interaction model
interaction_model = OrderedModel(
    outcome,
    X_country,
    distr="logit"
)

interaction_fit = interaction_model.fit(
    method="bfgs",
    disp=False,
    maxiter=1000
)

# Extract interaction results
interaction_coefficient = (
    interaction_fit.params["Dependency_x_Indonesia"]
)

interaction_ci = interaction_fit.conf_int().loc[
    "Dependency_x_Indonesia"
]

interaction_results = pd.DataFrame({
    "Measure": [
        "Interaction Odds Ratio",
        "95% CI Lower",
        "95% CI Upper",
        "Interaction p-value"
    ],
    "Value": [
        np.exp(interaction_coefficient),
        np.exp(interaction_ci[0]),
        np.exp(interaction_ci[1]),
        interaction_fit.pvalues[
            "Dependency_x_Indonesia"
        ]
    ]
})

print("COUNTRY INTERACTION RESULTS")
display(interaction_results.round(4))

# Likelihood-ratio test:
# original adjusted model vs interaction model
lr_stat = 2 * (
    interaction_fit.llf - result.llf
)

lr_p = chi2.sf(lr_stat, df=1)

print("\nLIKELIHOOD-RATIO INTERACTION TEST")
print("Chi-square:", round(lr_stat, 4))
print("p-value:", round(lr_p, 4))

print("\nMODEL CONVERGED:")
print(interaction_fit.mle_retvals.get("converged"))

# Country-specific dependency associations
# USA is the reference country

usa_or = np.exp(
    interaction_fit.params["AI_Dependency_Score"]
)

indonesia_or = np.exp(
    interaction_fit.params["AI_Dependency_Score"]
    + interaction_fit.params["Dependency_x_Indonesia"]
)

country_effects = pd.DataFrame({
    "Country": ["USA", "Indonesia"],
    "Estimated Odds Ratio": [
        usa_or,
        indonesia_or
    ]
})

print("\nCOUNTRY-SPECIFIC ESTIMATED ODDS RATIOS")
display(country_effects.round(4))


COUNTRY INTERACTION RESULTS


,Measure,Value
0,Interaction Odds Ratio,0.8407
1,95% CI Lower,0.4425
2,95% CI Upper,1.5973
3,Interaction p-value,0.5962



LIKELIHOOD-RATIO INTERACTION TEST
Chi-square: 0.2808
p-value: 0.5962

MODEL CONVERGED:
True

COUNTRY-SPECIFIC ESTIMATED ODDS RATIOS


,Country,Estimated Odds Ratio
0,USA,3.2063
1,Indonesia,2.6957


#### Country-specific odds ratios and 95% CIs

In [29]:

from scipy.stats import norm

# Coefficients
beta_dependency = interaction_fit.params[
    "AI_Dependency_Score"
]

beta_interaction = interaction_fit.params[
    "Dependency_x_Indonesia"
]

# Variance-covariance matrix
cov_matrix = interaction_fit.cov_params()

# USA: dependency coefficient
usa_beta = beta_dependency

usa_se = np.sqrt(
    cov_matrix.loc[
        "AI_Dependency_Score",
        "AI_Dependency_Score"
    ]
)

# Indonesia: dependency + interaction coefficient
indonesia_beta = beta_dependency + beta_interaction

indonesia_variance = (
    cov_matrix.loc[
        "AI_Dependency_Score",
        "AI_Dependency_Score"
    ]
    + cov_matrix.loc[
        "Dependency_x_Indonesia",
        "Dependency_x_Indonesia"
    ]
    + 2 * cov_matrix.loc[
        "AI_Dependency_Score",
        "Dependency_x_Indonesia"
    ]
)

indonesia_se = np.sqrt(indonesia_variance)

# Calculate odds ratios and confidence intervals
z = norm.ppf(0.975)

country_results = []

for country, beta, se in [
    ("USA", usa_beta, usa_se),
    ("Indonesia", indonesia_beta, indonesia_se)
]:
    country_results.append({
        "Country": country,
        "Odds Ratio": np.exp(beta),
        "CI Lower": np.exp(beta - z * se),
        "CI Upper": np.exp(beta + z * se)
    })

country_ci_table = pd.DataFrame(country_results)

print("COUNTRY-SPECIFIC ASSOCIATIONS WITH 95% CIs")
display(country_ci_table.round(4))


COUNTRY-SPECIFIC ASSOCIATIONS WITH 95% CIs


,Country,Odds Ratio,CI Lower,CI Upper
0,USA,3.2063,1.9929,5.1584
1,Indonesia,2.6957,1.6629,4.3699


#### Multicollinearity diagnostics



In [30]:

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Original adjusted model predictors
X_vif = model_data[
    [
        "AI_Dependency_Score",
        "Indonesia",
        "Study_Year",
        "AI_Frequency"
    ]
].astype(float)

# Include an intercept for VIF calculation
X_vif_constant = sm.add_constant(X_vif)

vif_results = pd.DataFrame({
    "Predictor": X_vif.columns,
    "VIF": [
        variance_inflation_factor(
            X_vif_constant.values,
            i + 1
        )
        for i in range(len(X_vif.columns))
    ]
})

print("MULTICOLLINEARITY DIAGNOSTICS")
display(vif_results.round(4))

print("\nCORRELATION MATRIX")
display(X_vif.corr().round(3))


MULTICOLLINEARITY DIAGNOSTICS


,Predictor,VIF
0,AI_Dependency_Score,1.1329
1,Indonesia,1.0538
2,Study_Year,1.0069
3,AI_Frequency,1.0835



CORRELATION MATRIX


,AI_Dependency_Score,Indonesia,Study_Year,AI_Frequency
AI_Dependency_Score,1.000,0.218,0.030,0.274
Indonesia,0.218,1.000,-0.053,0.051
Study_Year,0.030,-0.053,1.000,0.054
AI_Frequency,0.274,0.051,0.054,1.000


#### Publication-ready primary regression table


In [31]:
primary_variables = [
    "AI_Dependency_Score",
    "Indonesia",
    "Study_Year",
    "AI_Frequency"
]

variable_labels = {
    "AI_Dependency_Score": "AI Dependency Score",
    "Indonesia": "Indonesia (vs USA)",
    "Study_Year": "Year of Study",
    "AI_Frequency": "AI-use Frequency"
}

confidence_intervals = result.conf_int()

primary_table = []

for variable in primary_variables:

    odds_ratio = np.exp(result.params[variable])

    ci_lower = np.exp(
        confidence_intervals.loc[variable, 0]
    )

    ci_upper = np.exp(
        confidence_intervals.loc[variable, 1]
    )

    p_value = result.pvalues[variable]

    primary_table.append({
        "Predictor": variable_labels[variable],
        "Adjusted OR": round(odds_ratio, 3),
        "95% CI": (
            f"{ci_lower:.3f}–{ci_upper:.3f}"
        ),
        "p-value": (
            "<0.001" if p_value < 0.001
            else f"{p_value:.3f}"
        )
    })

primary_results_table = pd.DataFrame(primary_table)

print("TABLE 1: PRIMARY ORDINAL LOGISTIC REGRESSION")
display(primary_results_table)

print("\nMODEL INFORMATION")
print("Sample size:", len(outcome))
print("Log-likelihood:", round(result.llf, 3))
print("AIC:", round(result.aic, 3))
print("BIC:", round(result.bic, 3))
print(
    "Converged:",
    result.mle_retvals.get("converged")
)


TABLE 1: PRIMARY ORDINAL LOGISTIC REGRESSION


,Predictor,Adjusted OR,95% CI,p-value
0,AI Dependency Score,2.946,2.065–4.204,<0.001
1,Indonesia (vs USA),0.885,0.548–1.432,0.620
2,Year of Study,0.892,0.711–1.119,0.322
3,AI-use Frequency,0.735,0.518–1.042,0.084



MODEL INFORMATION
Sample size: 227
Log-likelihood: -333.007
AIC: 682.014
BIC: 709.413
Converged: True



#### Consolidated sensitivity and robustness table



In [32]:
# Dependency dimension sensitivity analyses
dimension_table = sensitivity_table.copy()

dimension_table["Analysis"] = (
    "Dependency dimension sensitivity"
)

dimension_table = dimension_table.rename(
    columns={
        "Predictor": "Predictor",
        "Odds Ratio": "Odds Ratio"
    }
)

dimension_table["Specification"] = (
    dimension_table["Model"]
)

# Alternative control-variable specifications
control_table = robustness_table.copy()

control_table["Analysis"] = (
    "Control-variable robustness"
)

control_table["Specification"] = (
    control_table["Model"]
)

control_table["Predictor"] = (
    "AI_Dependency_Score"
)

# Country-specific associations
country_table = country_ci_table.copy()

country_table["Analysis"] = (
    "Country interaction analysis"
)

country_table["Specification"] = (
    country_table["Country"]
)

country_table["Predictor"] = (
    "AI_Dependency_Score"
)

country_table["p-value"] = float("nan")

# Combine all results
columns = [
    "Analysis",
    "Specification",
    "Predictor",
    "Odds Ratio",
    "CI Lower",
    "CI Upper",
    "p-value"
]

final_sensitivity_table = pd.concat(
    [
        dimension_table[columns],
        control_table[columns],
        country_table[columns]
    ],
    ignore_index=True
)

# Format confidence intervals
final_sensitivity_table["95% CI"] = (
    final_sensitivity_table["CI Lower"].map(
        lambda x: f"{x:.3f}"
    )
    + "–"
    + final_sensitivity_table["CI Upper"].map(
        lambda x: f"{x:.3f}"
    )
)

# Format p-values
def format_p(p):
    if pd.isna(p):
        return "—"
    elif p < 0.001:
        return "<0.001"
    else:
        return f"{p:.3f}"

final_sensitivity_table["p-value"] = (
    final_sensitivity_table["p-value"].apply(
        format_p
    )
)

final_sensitivity_table["Odds Ratio"] = (
    final_sensitivity_table["Odds Ratio"].round(3)
)

final_sensitivity_table = final_sensitivity_table[
    [
        "Analysis",
        "Specification",
        "Predictor",
        "Odds Ratio",
        "95% CI",
        "p-value"
    ]
]

print("TABLE 2: SENSITIVITY AND ROBUSTNESS ANALYSES")
display(final_sensitivity_table)


TABLE 2: SENSITIVITY AND ROBUSTNESS ANALYSES


,Analysis,Specification,Predictor,Odds Ratio,95% CI,p-value
0,Dependency dimension sensitivity,Cognitive Preoccupation Only,Cognitive_Preoccupation,1.435,1.059–1.944,0.020
1,Dependency dimension sensitivity,Withdrawal Only,Withdrawal,2.778,2.077–3.716,<0.001
2,Dependency dimension sensitivity,Both Dimensions,Cognitive_Preoccupation,0.897,0.639–1.259,0.529
3,Dependency dimension sensitivity,Both Dimensions,Withdrawal,2.890,2.105–3.968,<0.001
4,Control-variable robustness,Unadjusted,AI_Dependency_Score,2.653,1.904–3.695,<0.001
5,Control-variable robustness,Original Adjusted,AI_Dependency_Score,2.946,2.065–4.204,<0.001
6,Control-variable robustness,Categorical Adjusted,AI_Dependency_Score,2.797,1.955–4.002,<0.001
7,Country interaction analysis,USA,AI_Dependency_Score,3.206,1.993–5.158,—
8,Country interaction analysis,Indonesia,AI_Dependency_Score,2.696,1.663–4.370,—


##### Leave-one-out influence sensitivity analysis

In [33]:

X_full = model_data[
    [
        "AI_Dependency_Score",
        "Indonesia",
        "Study_Year",
        "AI_Frequency"
    ]
].astype(float).reset_index(drop=True)

y_full = outcome.reset_index(drop=True)

leave_one_out_results = []

for i in range(len(y_full)):

    keep = np.arange(len(y_full)) != i

    X_subset = X_full.loc[keep].reset_index(drop=True)
    y_subset = y_full.loc[keep].reset_index(drop=True)

    loo_model = OrderedModel(
        y_subset,
        X_subset,
        distr="logit"
    )

    loo_fit = loo_model.fit(
        method="bfgs",
        disp=False,
        maxiter=500
    )

    leave_one_out_results.append({
        "Removed Observation": i + 1,
        "Odds Ratio": np.exp(
            loo_fit.params["AI_Dependency_Score"]
        ),
        "Converged": loo_fit.mle_retvals.get(
            "converged",
            False
        )
    })

loo_table = pd.DataFrame(leave_one_out_results)

print("LEAVE-ONE-OUT SENSITIVITY SUMMARY")

summary_table = pd.DataFrame({
    "Measure": [
        "Original adjusted OR",
        "Minimum leave-one-out OR",
        "Maximum leave-one-out OR",
        "Mean leave-one-out OR",
        "Number of models",
        "Models converged"
    ],
    "Value": [
        np.exp(result.params["AI_Dependency_Score"]),
        loo_table["Odds Ratio"].min(),
        loo_table["Odds Ratio"].max(),
        loo_table["Odds Ratio"].mean(),
        len(loo_table),
        loo_table["Converged"].sum()
    ]
})

display(summary_table.round(4))


LEAVE-ONE-OUT SENSITIVITY SUMMARY


,Measure,Value
0,Original adjusted OR,2.9462
1,Minimum leave-one-out OR,2.8580
2,Maximum leave-one-out OR,3.1286
3,Mean leave-one-out OR,2.9467
4,Number of models,227.0000
5,Models converged,227.0000



#### Final outcome and predictor distribution checks

In [34]:
# Outcome frequency distribution
outcome_distribution = (
    model_data["Problem_Solving_Impact"]
    .value_counts()
    .sort_index()
    .rename_axis("GAID7 Response")
    .reset_index(name="Frequency")
)

outcome_distribution["Percentage"] = (
    outcome_distribution["Frequency"]
    / outcome_distribution["Frequency"].sum()
    * 100
).round(2)

print("GAID7 OUTCOME DISTRIBUTION")
display(outcome_distribution)

# Dependency score descriptive statistics
dependency_summary = (
    model_data["AI_Dependency_Score"]
    .describe()
    .to_frame(name="AI Dependency Score")
)

print("\nAI DEPENDENCY SCORE SUMMARY")
display(dependency_summary.round(3))

# Distribution of dependency scores by outcome category
dependency_by_outcome = (
    model_data.groupby("Problem_Solving_Impact")[
        "AI_Dependency_Score"
    ]
    .agg(["count", "mean", "std", "median"])
    .round(3)
)

print("\nAI DEPENDENCY BY GAID7 RESPONSE CATEGORY")
display(dependency_by_outcome)


GAID7 OUTCOME DISTRIBUTION


,GAID7 Response,Frequency,Percentage
0,1.0,43,18.94
1,2.0,61,26.87
2,3.0,54,23.79
3,4.0,49,21.59
4,5.0,20,8.81



AI DEPENDENCY SCORE SUMMARY


,AI Dependency Score
count,227.000
mean,3.004
std,0.788
min,1.000
25%,2.429
50%,3.000
75%,3.571
max,4.857



AI DEPENDENCY BY GAID7 RESPONSE CATEGORY


,count,mean,std,median
Problem_Solving_Impact,,,,
1.0,43,2.505,0.739,2.571
2.0,61,2.881,0.715,2.857
3.0,54,3.156,0.684,3.143
4.0,49,3.204,0.784,3.429
5.0,20,3.550,0.786,3.500



#### Final measurement integrity check

In [35]:
dependency_items = [
    "GAID1", "GAID2", "GAID3",
    "GAID8", "GAID9", "GAID10", "GAID11"
]

excluded_items = ["GAID4", "GAID5", "GAID6", "GAID7"]

# Verify predictor and outcome do not share items
overlap = set(dependency_items).intersection(
    excluded_items
)

# Check item values
item_checks = []

for item in dependency_items + ["GAID7"]:

    values = pd.to_numeric(
        research[item],
        errors="coerce"
    )

    item_checks.append({
        "Item": item,
        "Minimum": values.min(),
        "Maximum": values.max(),
        "Missing": values.isna().sum(),
        "Invalid Responses": (
            ~values.isin([1, 2, 3, 4, 5])
        ).sum()
    })

print("ITEM VALIDITY CHECK")
display(pd.DataFrame(item_checks))

# Recalculate dependency composite
recalculated_score = research[
    dependency_items
].astype(float).mean(axis=1)

original_score = research[
    "AI_Dependency_Score"
].astype(float)

score_difference = (
    recalculated_score - original_score
).abs()

print("\nCOMPOSITE VERIFICATION")
print("Predictor/outcome item overlap:", overlap)
print(
    "Maximum absolute score difference:",
    round(score_difference.max(), 10)
)
print(
    "Number of mismatched scores:",
    int((score_difference > 1e-8).sum())
)
print(
    "Number of respondents checked:",
    len(research)
)


ITEM VALIDITY CHECK


,Item,Minimum,Maximum,Missing,Invalid Responses
0,GAID1,1.0,5.0,0,0
1,GAID2,1.0,5.0,0,0
2,GAID3,1.0,5.0,0,0
3,GAID8,1.0,5.0,0,0
4,GAID9,1.0,5.0,0,0
5,GAID10,1.0,5.0,0,0
6,GAID11,1.0,5.0,0,0
7,GAID7,1.0,5.0,0,0



COMPOSITE VERIFICATION
Predictor/outcome item overlap: set()
Maximum absolute score difference: 0.0
Number of mismatched scores: 0
Number of respondents checked: 227


#### Inspect GAID codebook and scoring documentation

In [36]:

print("CODEBOOK COLUMNS")
print(codebook.columns.tolist())

print("\nGAID-RELATED CODEBOOK ENTRIES")

# Search all columns for GAID references
gaid_mask = codebook.astype(str).apply(
    lambda col: col.str.contains(
        r"GAID|reverse.cod|scoring|dependency",
        case=False,
        regex=True,
        na=False
    )
).any(axis=1)

gaid_codebook = codebook.loc[gaid_mask]

display(gaid_codebook)

print("\nNUMBER OF MATCHING CODEBOOK ROWS:")
print(len(gaid_codebook))


CODEBOOK COLUMNS
['Variable', 'Description / Item wording', 'Construct', 'Subdimension', 'Response coding / Values', 'Scale source']

GAID-RELATED CODEBOOK ENTRIES


,Variable,Description / Item wording,Construct,Subdimension,Response coding / Values,Scale source
11,GAID1,1. My decisions are often influenced by generative AI.,GenAI Dependency,Cognitive Preoccupation,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"
12,GAID2,"2. I feel an urge to use generative AI, even when it may not be necessary.",GenAI Dependency,Cognitive Preoccupation,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"
13,GAID3,"3. I look forward to using generative AI, even if it is unrelated to the task at hand.",GenAI Dependency,Cognitive Preoccupation,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"
14,GAID4,4. My use of generative AI has caused concerns for me.,GenAI Dependency,Negative Consequences,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"
15,GAID5,5. I have trouble completing work or other responsibilities without generative AI.,GenAI Dependency,Negative Consequences,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"
16,GAID6,6. I feel less confident in my abilities without generative AI.,GenAI Dependency,Negative Consequences,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"
17,GAID7,7. My use of generative AI has negatively affected my problem-solving skills or efficiency.,GenAI Dependency,Negative Consequences,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"
18,GAID8,8. I experience restlessness if I am unable to use generative AI.,GenAI Dependency,Withdrawal,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"
19,GAID9,9. I feel unsettled or distracted when I cannot use generative AI.,GenAI Dependency,Withdrawal,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"
20,GAID10,10. I feel a sense of disconnection when I cannot access generative AI.,GenAI Dependency,Withdrawal,1 = Strongly disagree … 5 = Strongly agree,"Generative AI Dependency Scale (Goh, Hartanto & Majeed, 2025)"



NUMBER OF MATCHING CODEBOOK ROWS:
11


#### Final item-level reliability diagnostics

In [38]:

dependency_items = [
    "GAID1", "GAID2", "GAID3",
    "GAID8", "GAID9", "GAID10", "GAID11"
]

items = research[dependency_items].astype(float)

def cronbach_alpha(df):
    k = df.shape[1]
    item_variances = df.var(axis=0, ddof=1).sum()
    total_variance = df.sum(axis=1).var(ddof=1)

    return (
        k / (k - 1)
        * (1 - item_variances / total_variance)
    )

full_alpha = cronbach_alpha(items)

reliability_results = []

for item in dependency_items:

    remaining_items = items.drop(columns=[item])

    corrected_item_total = items[item].corr(
        remaining_items.sum(axis=1)
    )

    alpha_if_deleted = cronbach_alpha(
        remaining_items
    )

    reliability_results.append({
        "Item": item,
        "Item Mean": items[item].mean(),
        "Item SD": items[item].std(),
        "Corrected Item-Total Correlation":
            corrected_item_total,
        "Alpha if Deleted": alpha_if_deleted
    })

reliability_table = pd.DataFrame(reliability_results)

print("FINAL SEVEN-ITEM DEPENDENCY RELIABILITY")
print("Cronbach's Alpha:", round(full_alpha, 4))
print("Number of Items:", len(dependency_items))
print("Number of Respondents:", len(items))

print("\nITEM-LEVEL RELIABILITY DIAGNOSTICS")
display(reliability_table.round(4))


FINAL SEVEN-ITEM DEPENDENCY RELIABILITY
Cronbach's Alpha: 0.8009
Number of Items: 7
Number of Respondents: 227

ITEM-LEVEL RELIABILITY DIAGNOSTICS


,Item,Item Mean,Item SD,Corrected Item-Total Correlation,Alpha if Deleted
0,GAID1,3.4846,1.0490,0.4881,0.7829
1,GAID2,3.3524,1.1746,0.4699,0.7864
2,GAID3,3.4934,1.1497,0.4044,0.7975
3,GAID8,2.5991,1.2804,0.6182,0.7582
4,GAID9,2.7093,1.1727,0.6530,0.7524
5,GAID10,2.9427,1.1372,0.5621,0.7698
6,GAID11,2.4449,1.1975,0.5356,0.7745



#### Exploratory factor analysis of dependency items

In [40]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import (
    calculate_bartlett_sphericity,
    calculate_kmo
)

dependency_items = [
    "GAID1", "GAID2", "GAID3",
    "GAID8", "GAID9", "GAID10", "GAID11"
]

factor_data = research[
    dependency_items
].astype(float)

# Check suitability for factor analysis
chi_square_value, bartlett_p = (
    calculate_bartlett_sphericity(factor_data)
)

kmo_per_item, kmo_overall = (
    calculate_kmo(factor_data)
)

print("FACTOR ANALYSIS SUITABILITY")
print("Bartlett chi-square:", round(chi_square_value, 3))
print("Bartlett p-value:", bartlett_p)
print("Overall KMO:", round(kmo_overall, 3))

# One-factor solution
fa_one = FactorAnalyzer(
    n_factors=1,
    rotation=None,
    method="principal"
)

fa_one.fit(factor_data)

one_factor_loadings = pd.DataFrame(
    fa_one.loadings_,
    index=dependency_items,
    columns=["Factor 1"]
)

print("\nONE-FACTOR LOADINGS")
display(one_factor_loadings.round(3))

# Two-factor solution with oblique rotation
fa_two = FactorAnalyzer(
    n_factors=2,
    rotation="promax",
    method="principal"
)

fa_two.fit(factor_data)

two_factor_loadings = pd.DataFrame(
    fa_two.loadings_,
    index=dependency_items,
    columns=["Factor 1", "Factor 2"]
)

print("\nTWO-FACTOR LOADINGS")
display(two_factor_loadings.round(3))

print("\nTWO-FACTOR CORRELATION MATRIX")
display(
    pd.DataFrame(
        fa_two.phi_,
        columns=["Factor 1", "Factor 2"],
        index=["Factor 1", "Factor 2"]
    ).round(3)
)

# Eigenvalues from item correlation matrix
eigenvalues, _ = fa_two.get_eigenvalues()

print("\nEIGENVALUES")
display(
    pd.DataFrame({
        "Factor": range(1, len(eigenvalues) + 1),
        "Eigenvalue": eigenvalues
    }).round(3)
)


FACTOR ANALYSIS SUITABILITY
Bartlett chi-square: 446.736
Bartlett p-value: 1.8732464120175523e-81
Overall KMO: 0.824


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



ONE-FACTOR LOADINGS


,Factor 1
GAID1,-0.616
GAID2,-0.602
GAID3,-0.531
GAID8,-0.766
GAID9,-0.789
GAID10,-0.715
GAID11,-0.690



TWO-FACTOR LOADINGS


...

,Factor 1,Factor 2
GAID1,0.090,0.709
GAID2,0.063,0.725
GAID3,-0.136,0.874
GAID8,0.789,0.046
GAID9,0.796,0.068
GAID10,0.796,-0.030
GAID11,0.802,-0.071



TWO-FACTOR CORRELATION MATRIX


,Factor 1,Factor 2
Factor 1,1.000,0.447
Factor 2,0.447,1.000



EIGENVALUES


,Factor,Eigenvalue
0,1,3.219
1,2,1.151
2,3,0.652
3,4,0.590
4,5,0.535
5,6,0.500
6,7,0.353



#### Parallel analysis for factor retention

In [41]:
factor_items = [
    "GAID1", "GAID2", "GAID3",
    "GAID8", "GAID9", "GAID10", "GAID11"
]

factor_values = research[
    factor_items
].astype(float).to_numpy()

n_rows, n_columns = factor_values.shape

# Observed correlation matrix eigenvalues
observed_corr = np.corrcoef(
    factor_values,
    rowvar=False
)

observed_eigenvalues = np.linalg.eigvalsh(
    observed_corr
)[::-1]

# Generate reference eigenvalues from random data
rng = np.random.default_rng(42)

n_simulations = 1000

random_eigenvalues = np.zeros(
    (n_simulations, n_columns)
)

for i in range(n_simulations):

    random_data = rng.normal(
        size=(n_rows, n_columns)
    )

    random_corr = np.corrcoef(
        random_data,
        rowvar=False
    )

    random_eigenvalues[i] = np.linalg.eigvalsh(
        random_corr
    )[::-1]

# 95th percentile of random eigenvalues
random_95th = np.percentile(
    random_eigenvalues,
    95,
    axis=0
)

parallel_results = pd.DataFrame({
    "Factor": range(1, n_columns + 1),
    "Observed Eigenvalue": observed_eigenvalues,
    "Random 95th Percentile": random_95th,
    "Retain Factor": (
        observed_eigenvalues > random_95th
    )
})

print("PARALLEL ANALYSIS RESULTS")
display(parallel_results.round(3))

print("\nNUMBER OF FACTORS SUGGESTED:")
print(
    int(parallel_results["Retain Factor"].sum())
)


PARALLEL ANALYSIS RESULTS


,Factor,Observed Eigenvalue,Random 95th Percentile,Retain Factor
0,1,3.219,1.348,True
1,2,1.151,1.211,False
2,3,0.652,1.118,False
3,4,0.590,1.040,False
4,5,0.535,0.970,False
5,6,0.500,0.904,False
6,7,0.353,0.829,False



NUMBER OF FACTORS SUGGESTED:
1
